# Laboratorio 03 — Funciones Avanzadas libres con PySpark

**Semana:** 02  
**Actividad de referencia:** Actividad 03  
**Estudiante:** Daniel Guzmán  
**Dataset:** Cards Data — Financial Transaction Dataset  

## Parte 1 — Descripción del dataset

### Nombre y fuente

El dataset seleccionado es `cards_data.csv`, parte del **Financial Transaction Dataset** usado durante la Semana 02 del bootcamp.

Fuente: SharePoint del bootcamp  
Ruta: `inetum_data_engineer_bootcamp / semana_02 / financial_transaction_dataset`

### Dominio

El dominio del dataset es **financiero/bancario**.  
Describe tarjetas asociadas a clientes, incluyendo marca, tipo, límite de crédito, chip, fecha de expiración y fecha de apertura de cuenta.

### ¿Por qué elegí este dataset?

Elegí este dataset porque ya fue explorado en los laboratorios anteriores y tiene una columna temporal que permite analizar la evolución de tarjetas en el tiempo.

La pregunta temporal principal es entender cómo cambian los límites de crédito según el año o mes de apertura de la tarjeta.

### Columna temporal

La columna temporal principal es `acct_open_date`.

Su granularidad es mensual, porque viene en formato `MM/yyyy`.

### Columna numérica principal

La métrica principal será `credit_limit`, convertida a número como `credit_limit_num`.

### Columna categórica para particionar

Para las window functions se usará principalmente `card_brand`, porque permite comparar tarjetas dentro de cada marca.

### Preguntas de negocio temporales

1. ¿En qué años se abrieron más tarjetas?
2. ¿Qué marca de tarjeta tuvo mayor límite promedio por año de apertura?
3. ¿Cómo evoluciona el límite de crédito acumulado por marca a lo largo del tiempo?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import col, when, isnan, sum as spark_sum

MI_NOMBRE = "daniel"
VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"
ARCHIVO = "cards_data.csv"

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOL}/{ARCHIVO}")

print(f"Filas: {df_raw.count():,} | Columnas: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
df_raw.describe().show(truncate=False)

In [0]:
total = df_raw.count()
numeric_types = {"double", "float", "long", "integer", "short", "byte"}

nulos = df_raw.select([
    spark_sum(
        when(
            col(c).isNull() |
            (isnan(col(c)) if df_raw.schema[c].dataType.typeName() in numeric_types else F.lit(False)) |
            (col(c).cast("string") == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_raw.columns
]).collect()[0].asDict()

print(f"{'Columna':<35} {'Nulos':>8} {'%':>8}")
print("-" * 54)

for c, n in sorted(nulos.items(), key=lambda x: -x[1]):
    print(f"{c:<35} {n:>8,} {n/total*100:>7.1f}%")

## Observaciones del perfil

El dataset tiene 6,146 registros y 13 columnas, por lo que cumple el requisito mínimo de más de 5,000 filas.

La columna temporal principal es `acct_open_date`, que representa la fecha de apertura de la cuenta o tarjeta. Para poder usar funciones temporales será necesario convertirla a tipo fecha.

La columna numérica principal será `credit_limit`, pero primero debe limpiarse para remover símbolos de moneda o separadores y convertirse a tipo `double`.

Antes de aplicar window functions se validaron nulos y tipos de datos para evitar errores en los análisis temporales.

## Observaciones del perfil

El dataset tiene 6,146 registros y 13 columnas, por lo que cumple el requisito mínimo de más de 5,000 filas.

La columna temporal principal es `acct_open_date`, que representa la fecha de apertura de la cuenta o tarjeta. Para poder usar funciones temporales será necesario convertirla a tipo fecha.

La columna numérica principal será `credit_limit`, pero primero debe limpiarse para remover símbolos de moneda o separadores y convertirse a tipo `double`.

Antes de aplicar window functions se validaron nulos y tipos de datos para evitar errores en los análisis temporales.

In [0]:
# Parte 3 — Preparación de la columna temporal

COL_FECHA = "acct_open_date"
COL_NUMERICA = "credit_limit_num"
COL_CATEGORIA = "card_brand"

df = (
    df_raw
    .withColumn("credit_limit_num", F.regexp_replace(F.col("credit_limit"), "[$,]", "").cast("double"))
    .withColumn("ts", F.to_date(F.col(COL_FECHA), "MM/yyyy"))
    .withColumn("mes", F.month("ts"))
    .withColumn("anio", F.year("ts"))
    .withColumn("semana_anio", F.weekofyear("ts"))
    .withColumn("dia_semana", F.dayofweek("ts"))
    .withColumn("mes_inicio", F.date_trunc("month", "ts"))
)

df.select(
    "acct_open_date",
    "ts",
    "mes",
    "anio",
    "semana_anio",
    "dia_semana",
    "mes_inicio",
    "card_brand",
    "card_type",
    "credit_limit",
    "credit_limit_num"
).show(10, truncate=False)

## Granularidad temporal

La columna `acct_open_date` tiene granularidad mensual, porque viene en formato `MM/yyyy`.

Por esta razón, los análisis tienen más sentido a nivel de **mes** o **año**, no a nivel de hora o minuto. Para este laboratorio se usará principalmente el año de apertura (`anio`) y el inicio del mes (`mes_inicio`) para analizar evolución temporal.

In [0]:
df.select(
    F.count("*").alias("total_registros"),
    F.sum(F.when(F.col("ts").isNull(), 1).otherwise(0)).alias("fechas_invalidas"),
    F.sum(F.when(F.col("credit_limit_num").isNull(), 1).otherwise(0)).alias("limites_invalidos")
).show()

In [0]:
# Parte 4 — Agregaciones avanzadas por período
# Evolución por año de apertura

df_anual = (
    df
    .groupBy("anio")
    .agg(
        F.count("*").alias("tarjetas_abiertas"),
        F.round(F.sum(COL_NUMERICA), 2).alias("limite_total"),
        F.round(F.avg(COL_NUMERICA), 2).alias("limite_promedio"),
        F.percentile_approx(COL_NUMERICA, 0.9).alias("percentil_90_limite")
    )
    .orderBy("anio")
)

df_anual.show(50, truncate=False)

## Observaciones de agregación anual

Se agruparon las tarjetas por año de apertura para analizar cuántas tarjetas se abrieron en cada periodo y cómo evolucionó el límite de crédito.

Las métricas usadas fueron:

- Total de tarjetas abiertas.
- Límite total.
- Límite promedio.
- Percentil 90 del límite de crédito.

Este análisis permite detectar años con mayor emisión de tarjetas y años donde los límites promedio fueron más altos.

In [0]:
# Evolución anual por marca de tarjeta

df_anual_marca = (
    df
    .groupBy("anio", "card_brand")
    .agg(
        F.count("*").alias("tarjetas_abiertas"),
        F.round(F.avg(COL_NUMERICA), 2).alias("limite_promedio"),
        F.max(COL_NUMERICA).alias("limite_maximo")
    )
    .orderBy("anio", "card_brand")
)

df_anual_marca.show(50, truncate=False)

## Observaciones por marca y año

Se agregó una segunda agrupación combinando `anio` y `card_brand`.

Esto permite comparar si las marcas de tarjeta tienen comportamientos distintos en el tiempo. En un análisis financiero, esta vista ayuda a identificar qué marcas concentran límites más altos o mayor volumen de tarjetas emitidas por año.

In [0]:
# Parte 5 — Window Function: Ranking
# Ranking de tarjetas con mayor límite dentro de cada marca

windowSpec_rank = Window.partitionBy(COL_CATEGORIA).orderBy(F.col(COL_NUMERICA).desc())

df_ranking = df.withColumn("rank_en_marca", F.rank().over(windowSpec_rank))

df_ranking.filter(F.col("rank_en_marca") <= 3) \
    .orderBy(COL_CATEGORIA, "rank_en_marca") \
    .select(
        COL_CATEGORIA,
        "card_type",
        "credit_limit",
        COL_NUMERICA,
        "acct_open_date",
        "ts",
        "rank_en_marca"
    ) \
    .show(30, truncate=False)

## Window Function — Ranking

Para el ranking se usó:

- `PARTITION BY card_brand`
- `ORDER BY credit_limit_num DESC`

Esto permite encontrar las tarjetas con mayor límite de crédito dentro de cada marca, sin mezclar marcas diferentes.

`rank()` asigna la misma posición a registros empatados y deja saltos en el ranking.  
`dense_rank()` también asigna la misma posición a empates, pero no deja saltos.

En este caso, `rank()` tiene sentido porque si varias tarjetas tienen el mismo límite dentro de una marca, deben compartir la misma posición.

In [0]:
# Parte 6 — Window Function: LAG y detección de variaciones
# Comparar cada tarjeta contra la tarjeta anterior dentro de la misma marca, ordenada por fecha de apertura

windowSpec_lag = Window.partitionBy(COL_CATEGORIA).orderBy("ts", "id")

df = (
    df
    .withColumn("valor_anterior", F.lag(COL_NUMERICA, 1).over(windowSpec_lag))
    .withColumn(
        "variacion",
        F.round(F.col(COL_NUMERICA) - F.col("valor_anterior"), 2)
    )
)

df.filter(F.col("variacion").isNotNull()) \
    .orderBy(F.col("variacion").desc()) \
    .select(
        COL_CATEGORIA,
        "card_type",
        "ts",
        COL_NUMERICA,
        "valor_anterior",
        "variacion"
    ) \
    .show(10, truncate=False)

## Window Function — LAG y variaciones

Se usó `lag()` para comparar el límite de crédito de una tarjeta contra el límite de la tarjeta anterior dentro de la misma marca (`card_brand`), ordenando por fecha de apertura.

Las variaciones más grandes representan saltos importantes entre una tarjeta y la anterior en la misma marca. Estos saltos pueden deberse a diferencias entre perfiles de clientes, cambios en políticas de crédito o simplemente a que el dataset no representa una serie temporal continua por cliente.

En este caso, la variación no debe interpretarse automáticamente como anomalía, porque estamos comparando tarjetas diferentes dentro de una marca, no necesariamente el historial de una misma tarjeta o de un mismo cliente.

In [0]:
# Parte 7 — Window Function: Acumulado
# Límite de crédito acumulado por marca a lo largo del tiempo

windowSpec_acum = (
    Window
    .partitionBy(COL_CATEGORIA)
    .orderBy("ts", "id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df = df.withColumn(
    "limite_acumulado_marca",
    F.round(F.sum(COL_NUMERICA).over(windowSpec_acum), 2)
)

df.select(
    COL_CATEGORIA,
    "card_type",
    "ts",
    COL_NUMERICA,
    "limite_acumulado_marca"
).orderBy(COL_CATEGORIA, "ts").show(30, truncate=False)

## Window Function — Acumulado

Elegí la opción de acumulado porque el dataset permite analizar cómo crece el límite total de crédito por marca a lo largo del tiempo.

La ventana se particionó por `card_brand` y se ordenó por `ts`, que representa la fecha de apertura de la cuenta o tarjeta.

El acumulado permite ver cómo cada nueva tarjeta va sumando al total de exposición crediticia de cada marca. Esto puede ser útil para entender la evolución del portafolio financiero.

In [0]:
# Pregunta 1: ¿En qué años se abrieron más tarjetas?

df_tarjetas_por_anio = (
    df
    .groupBy("anio")
    .agg(
        F.count("*").alias("tarjetas_abiertas"),
        F.round(F.avg(COL_NUMERICA), 2).alias("limite_promedio"),
        F.round(F.sum(COL_NUMERICA), 2).alias("limite_total")
    )
    .orderBy(F.col("tarjetas_abiertas").desc())
)

df_tarjetas_por_anio.show(10, truncate=False)

## Conclusión pregunta 1

El año con mayor apertura de tarjetas fue **2020**, con **1,178 tarjetas abiertas**.

Le siguen **2010**, con **545 tarjetas**, y **2008**, con **487 tarjetas**. Esto muestra un pico fuerte de apertura en 2020 frente a los demás años.

Además, aunque 2020 tuvo el mayor volumen de tarjetas, su límite promedio fue **13,968.94**, mientras que años con menos tarjetas como **2013** tuvieron un límite promedio mayor (**17,698.71**). Esto indica que más tarjetas abiertas no necesariamente significa mayor límite promedio.

In [0]:
# Pregunta 2: ¿Qué marca tuvo mayor límite promedio por año?

df_marca_anio = (
    df
    .groupBy("anio", "card_brand")
    .agg(
        F.count("*").alias("tarjetas_abiertas"),
        F.round(F.avg(COL_NUMERICA), 2).alias("limite_promedio")
    )
)

window_marca_anio = (
    Window
    .partitionBy("anio")
    .orderBy(F.col("limite_promedio").desc())
)

df_top_marca_anio = (
    df_marca_anio
    .withColumn("rank_anio", F.rank().over(window_marca_anio))
    .filter(F.col("rank_anio") == 1)
    .orderBy("anio")
)

df_top_marca_anio.show(50, truncate=False)

## Conclusión pregunta 2

El liderazgo por límite promedio cambia según el año.

En varios años aparece **Mastercard** como la marca con mayor límite promedio, por ejemplo en **2001**, **2002**, **2004**, **2005**, **2012**, **2013**, **2015**, **2016** y **2020**.

También aparecen años donde lidera **Visa**, como **2000**, **2003**, **2006**, **2007**, **2008**, **2011**, **2014** y **2017**.

En 2018 y 2019 aparece **Discover** como líder, pero con solo **1 tarjeta** en cada año, por lo que ese resultado debe interpretarse con cuidado porque puede no ser representativo.

In [0]:
# Pregunta 3: ¿Cómo evoluciona el límite de crédito acumulado por marca?

df_acumulado_resumen = (
    df
    .groupBy("card_brand", "anio")
    .agg(
        F.count("*").alias("tarjetas_abiertas"),
        F.round(F.sum(COL_NUMERICA), 2).alias("limite_anual")
    )
)

window_acum_anual = (
    Window
    .partitionBy("card_brand")
    .orderBy("anio")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_acumulado_anual = (
    df_acumulado_resumen
    .withColumn(
        "limite_acumulado",
        F.round(F.sum("limite_anual").over(window_acum_anual), 2)
    )
    .orderBy("card_brand", "anio")
)

df_acumulado_anual.show(100, truncate=False)

In [0]:
# Resumen final del límite acumulado por marca

window_final = Window.partitionBy("card_brand").orderBy(F.col("anio").desc())

df_acumulado_final = (
    df_acumulado_anual
    .withColumn("rn", F.row_number().over(window_final))
    .filter(F.col("rn") == 1)
    .select("card_brand", "anio", "limite_acumulado")
    .orderBy(F.col("limite_acumulado").desc())
)

df_acumulado_final.show(truncate=False)

## Conclusión pregunta 3

El límite acumulado permite ver cómo crece la exposición crediticia por marca a través del tiempo.

Al cierre del periodo, la marca con mayor límite acumulado fue **Mastercard**, con aproximadamente **47,042,657**. Le sigue **Visa**, con aproximadamente **34,279,041**, y luego **Amex**, con **4,597,400**. **Discover** queda muy por debajo, con **226,060**.

Esto muestra que Mastercard concentra la mayor exposición crediticia acumulada dentro del dataset. La window function acumulada es útil porque permite conservar la evolución por año y, al mismo tiempo, calcular el crecimiento acumulado dentro de cada marca.

## Parte 9 — Reflexión final

### ¿Cuál window function te resultó más difícil de entender o aplicar? ¿Por qué?

La window function más difícil fue el acumulado, porque requiere entender la combinación entre `partitionBy`, `orderBy` y `rowsBetween`.

No es solo agrupar datos, sino calcular un valor progresivo dentro de cada grupo, manteniendo el orden temporal.

### ¿En qué se diferencia una window function de un groupBy para responder la misma pregunta?

Un `groupBy` reduce las filas y devuelve un resumen por grupo. En cambio, una window function permite calcular métricas dentro de una partición sin perder necesariamente el detalle de las filas.

Por ejemplo, con `groupBy` puedo calcular el límite total por marca y año, pero con una window function puedo calcular el acumulado progresivo por marca a través del tiempo.

### ¿Qué hallazgo temporal te sorprendió más en tu dataset?

El hallazgo más llamativo fue que **2020** fue el año con mayor apertura de tarjetas, con **1,178 tarjetas**, muy por encima de otros años.

También fue interesante ver que la marca con mayor límite acumulado fue **Mastercard**, con más de **47 millones** de límite acumulado.

### ¿Qué análisis adicional harías si tuvieras más columnas o más historia temporal?

Haría un análisis cruzando tarjetas con transacciones para revisar si los años de apertura, la marca de tarjeta o el límite de crédito están relacionados con mayor volumen transaccional o mayor riesgo de fraude.

También sería útil analizar cambios por cliente a lo largo del tiempo, pero para eso se necesitaría una serie temporal más detallada por usuario o por tarjeta.